# Results reviewer

Coal installed-capacity diagnostics and map plots for one EMPIRE scenario
(folder `Data_handler_energy_vis_<Scenario>`, e.g. Trinity, REPowerEU++,
GoRES, NECPEssentials). Plot types and logic are unchanged from the original
version of this notebook — only the input file locations were adapted to
match how `run_empire.py` writes its outputs directly into the scenario
result folder (no `Input`/`Output` subfolders for the result CSVs).

In [ ]:
# ── Dataset configuration ────────────────────────────────────────────
# Set SCENARIO_NAME to one of the EMPIRE scenario folders produced for this
# analysis. Each folder is the result_file_path used by run_empire.py and
# contains the results_output_*.csv files directly (no 'Output' subfolder).   energy_vis_
from pathlib import Path

BASE_PREFIX = "dataset_Data_handler_"
SCENARIO_NAME = "Trinity"   # <-- CHANGE ME: Trinity, REPowerEU++, GoRES, or NECPEssentials

RESULTS_BASE = Path(".")
DATASET_PATH = RESULTS_BASE / f"{BASE_PREFIX}{SCENARIO_NAME}"  # <-- CHANGE ME if folders are elsewhere

# Coordinates (Sets.xlsx) and offshore-node definitions (Sets_OffshoreNode.tab)
# are model INPUT files (tab_file_path in run_empire.py), not part of the
# result_file_path folder above, so they are configured separately here.
COORDS_FILE = Path("Sets.xlsx")                       # <-- CHANGE ME: path to the input Sets.xlsx coordinates file
OFFSHORE_TAB_FILE = Path("Sets_OffshoreNode.tab")     # <-- CHANGE ME: path to the input Sets_OffshoreNode.tab file

# Create plots output directory next to the dataset results
PLOT_DIR = DATASET_PATH / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Scenario : {SCENARIO_NAME}')
print(f'Results  : {DATASET_PATH.resolve()}')
print(f'Plots    : {PLOT_DIR.resolve()}')


In [ ]:
from pathlib import Path
import re
import pandas as pd

def period_end_year(period_label: str) -> int:
    years = re.findall(r"\d{4}", str(period_label))
    return int(years[-1]) if years else -1

results_path = DATASET_PATH / 'Output' / 'results_output_gen.csv'
df = pd.read_csv(results_path)

required = {'Node', 'GeneratorType', 'Period', 'genInstalledCap_MW'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing columns in {results_path}: {sorted(missing)}')

# Keep coal-related technologies (coal, lignite, hard-coal naming variants).
coal_mask = df['GeneratorType'].astype(str).str.contains(r'coal|lignite|hcoal|hardcoal', case=False, regex=True)
coal_df = df.loc[coal_mask].copy()

if coal_df.empty:
    print('No coal-related technologies found in results_output_gen.csv')
else:
    coal_df['genInstalledCap_MW'] = pd.to_numeric(coal_df['genInstalledCap_MW'], errors='coerce').fillna(0.0)

    period_order = sorted(coal_df['Period'].astype(str).unique().tolist(), key=period_end_year)
    p_first = period_order[0]
    p_last = period_order[-1]

    print('Coal-related technologies detected:')
    print(sorted(coal_df['GeneratorType'].dropna().unique().tolist()))

    print('\n1) Installed capacity by coal technology and period [MW]')
    by_tech_period = (
        coal_df.groupby(['GeneratorType', 'Period'], as_index=False)['genInstalledCap_MW']
        .sum()
    )
    by_tech_period['period_sort'] = by_tech_period['Period'].map(period_end_year)
    by_tech_period = by_tech_period.sort_values(['GeneratorType', 'period_sort', 'Period']).drop(columns=['period_sort'])
    display(by_tech_period)

    print('\n2) Aggregated coal installed capacity (all coal technologies) by period [MW]')
    agg_period = (
        coal_df.groupby('Period', as_index=False)['genInstalledCap_MW']
        .sum()
        .rename(columns={'genInstalledCap_MW': 'coal_total_installed_MW'})
    )
    agg_period['period_sort'] = agg_period['Period'].map(period_end_year)
    agg_period = agg_period.sort_values(['period_sort', 'Period']).drop(columns=['period_sort'])
    display(agg_period)

    print('\n3) Aggregated coal installed capacity by node and period [MW] (top 50 rows)')
    by_node_period = (
        coal_df.groupby(['Node', 'Period'], as_index=False)['genInstalledCap_MW']
        .sum()
        .rename(columns={'genInstalledCap_MW': 'coal_total_installed_MW'})
    )
    by_node_period['period_sort'] = by_node_period['Period'].map(period_end_year)
    by_node_period = by_node_period.sort_values(['period_sort', 'coal_total_installed_MW'], ascending=[True, False]).drop(columns=['period_sort'])
    display(by_node_period.head(50))

    print('\n4) Change from first to last period by coal technology [MW]')
    first_last = by_tech_period[by_tech_period['Period'].isin([p_first, p_last])].copy()
    first_last = first_last.pivot(index='GeneratorType', columns='Period', values='genInstalledCap_MW').fillna(0.0)
    if p_first not in first_last.columns:
        first_last[p_first] = 0.0
    if p_last not in first_last.columns:
        first_last[p_last] = 0.0
    first_last['delta_MW_last_minus_first'] = first_last[p_last] - first_last[p_first]
    first_last = first_last.reset_index().sort_values('delta_MW_last_minus_first', ascending=False)
    display(first_last)


In [ ]:
import matplotlib.pyplot as plt

if coal_df.empty:
    print('No coal data available for plotting.')
else:
    # Total coal change by node: last period - first period
    coal_by_node_period = (
        coal_df.groupby(['Node', 'Period'], as_index=False)['genInstalledCap_MW']
        .sum()
        .rename(columns={'genInstalledCap_MW': 'coal_total_installed_MW'})
    )

    period_order = sorted(coal_by_node_period['Period'].astype(str).unique().tolist(), key=period_end_year)
    p_first = period_order[0]
    p_last = period_order[-1]

    node_change = (
        coal_by_node_period[coal_by_node_period['Period'].isin([p_first, p_last])]
        .pivot(index='Node', columns='Period', values='coal_total_installed_MW')
        .fillna(0.0)
    )
    if p_first not in node_change.columns:
        node_change[p_first] = 0.0
    if p_last not in node_change.columns:
        node_change[p_last] = 0.0
    node_change['delta_MW_last_minus_first'] = node_change[p_last] - node_change[p_first]
    node_change = node_change.sort_values('delta_MW_last_minus_first', ascending=False)

    display(node_change[['delta_MW_last_minus_first']].reset_index())

    plt.figure(figsize=(16, 6))
    colors = ['#1a9850' if v >= 0 else '#d73027' for v in node_change['delta_MW_last_minus_first']]
    plt.bar(node_change.index, node_change['delta_MW_last_minus_first'], color=colors)
    plt.axhline(0.0, color='black', linewidth=1.0)
    plt.xticks(rotation=90)
    plt.ylabel('Coal capacity change [MW]')
    plt.title(f'Total coal capacity change by node ({p_first} -> {p_last})')
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'coal_capacity_change_by_node.png', dpi=150, bbox_inches='tight')
    plt.show()


## Map Plots

Installed-capacity pies, capacity-change bars, and expected load-curtailment bubbles for `dataset_north_sea`, adapted from `plot.py` for inline display.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Wedge
from shapely.geometry import shape

%matplotlib inline

# ── Paths ──────────────────────────────────────────────────
RESULTS_PATH = DATASET_PATH
BASEMAP_FILE = Path('NUTS_RG_60M_2024_3035.geojson')

# ── Aggregation flag ───────────────────────────────────────
AGGREGATE_OFFSHORE_TO_CLOSEST_ONSHORE = True

# ── Technology display order & colours ────────────────────
TECH_ORDER = ["Lignite","Hcoal","Coal","Gas","Oil","Bio","Geo","Nuclear","Hydro","Wave","Wind","Solar","Waste"]
TECH_COLORS = {
    "Lignite": "#7f7f7f", "Hcoal": "#4f4f4f",  "Coal":    "#9a9a9a",
    "Gas":     "#d62728", "Oil":   "#8c564b",   "Bio":     "#2ca02c",
    "Geo":     "#ff9896", "Nuclear":"#e377c2",  "Hydro":   "#17becf",
    "Wave":    "#1f9ed4", "Wind":  "#1f77b4",   "Solar":   "#ffbb00",
    "Waste":   "#bcbd22",
}
CHANGE_GROUPS = {
    "renewable_generation":                ["Bio","Geo","Hydro","Wave","Wind","Solar","Waste"],
    "thermal_generation_including_nuclear": ["Lignite","Hcoal","Coal","Gas","Oil","Nuclear"],
    "coal":                                ["Lignite","Hcoal","Coal"],
    "wind_and_solar":                      ["Wind","Solar"],
    "hydro":                               ["Hydro"],
}
GEOJSON_SOURCE_CRS = ccrs.epsg(3035)


In [ ]:
# ── Helper functions (from plot.py) ───────────────────────

def normalize_name(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def load_coords(coords_file: Path) -> pd.DataFrame:
    coords = pd.read_excel(coords_file, sheet_name="Coords", skiprows=2, usecols=[0, 1, 2])
    coords.columns = ["Location", "Latitude", "Longitude"]
    coords = coords.dropna(subset=["Location", "Latitude", "Longitude"]).copy()
    coords["norm_node"] = coords["Location"].map(normalize_name)
    return coords


def load_offshore_nodes(offshore_tab_file: Path) -> set[str]:
    offshore_file = offshore_tab_file
    if not offshore_file.exists():
        return set()
    offshore = pd.read_csv(offshore_file, sep="\t", header=None, names=["Node"])
    offshore["Node"] = offshore["Node"].astype(str).str.strip()
    offshore = offshore[offshore["Node"] != ""].copy()
    offshore["norm_node"] = offshore["Node"].map(normalize_name)
    offshore = offshore[offshore["norm_node"] != "offshorenode"]
    return set(offshore["norm_node"].tolist())


def build_offshore_to_onshore_map(coords: pd.DataFrame, offshore_norm_nodes: set[str]) -> dict[str, str]:
    if not offshore_norm_nodes:
        return {}
    coord_lookup = coords[["norm_node", "Latitude", "Longitude"]].drop_duplicates("norm_node")
    onshore_coords = coord_lookup.loc[~coord_lookup["norm_node"].isin(offshore_norm_nodes)].copy()
    offshore_coords = coord_lookup.loc[coord_lookup["norm_node"].isin(offshore_norm_nodes)].copy()
    if onshore_coords.empty or offshore_coords.empty:
        return {}
    nearest_map: dict[str, str] = {}
    for _, row in offshore_coords.iterrows():
        d2 = (onshore_coords["Latitude"] - row["Latitude"]) ** 2 + (onshore_coords["Longitude"] - row["Longitude"]) ** 2
        nearest_idx = d2.idxmin()
        nearest_map[str(row["norm_node"])] = str(onshore_coords.loc[nearest_idx, "norm_node"])
    return nearest_map


def aggregate_offshore_to_nearest_onshore(
    capacity: pd.DataFrame,
    coords: pd.DataFrame,
    offshore_norm_nodes: set[str],
    offshore_to_onshore: dict[str, str] | None = None,
) -> pd.DataFrame:
    if not offshore_norm_nodes:
        return capacity
    coord_lookup = coords[["norm_node", "Location", "Latitude", "Longitude"]].drop_duplicates("norm_node")
    if offshore_to_onshore is None:
        offshore_to_onshore = build_offshore_to_onshore_map(coords, offshore_norm_nodes)
    if not offshore_to_onshore:
        return capacity
    cap = capacity.copy()
    cap["target_norm_node"] = cap["norm_node"].map(offshore_to_onshore).fillna(cap["norm_node"])
    grouped = cap.groupby("target_norm_node", as_index=False)[TECH_ORDER].sum()
    grouped = grouped.rename(columns={"target_norm_node": "norm_node"})
    grouped["total_cap"] = grouped[TECH_ORDER].sum(axis=1)
    name_map = coord_lookup.set_index("norm_node")["Location"].to_dict()
    grouped["Node"] = grouped["norm_node"].map(name_map).fillna(grouped["norm_node"])
    return grouped[["Node", *TECH_ORDER, "total_cap", "norm_node"]]


def load_transmission_by_period(results_path: Path) -> tuple[dict[str, pd.DataFrame], list[str]]:
    """Load installed transmission capacity per period from results_output_transmision.csv."""
    csv_file = results_path / "results_output_transmision.csv"
    lines = pd.read_csv(csv_file)
    required = {"BetweenNode", "AndNode", "Period", "transmissionInstalledCap_MW"}
    missing = required - set(lines.columns)
    if missing:
        raise ValueError(f"Missing columns in {csv_file}: {sorted(missing)}")

    lines["transmissionInstalledCap_MW"] = pd.to_numeric(lines["transmissionInstalledCap_MW"], errors="coerce").fillna(0.0)
    lines = lines.dropna(subset=["BetweenNode", "AndNode", "Period"]).copy()
    lines["Period"] = lines["Period"].astype(str)
    lines["from_norm"] = lines["BetweenNode"].map(normalize_name)
    lines["to_norm"]   = lines["AndNode"].map(normalize_name)

    # Canonical undirected key to avoid plotting both A→B and B→A.
    lines["u"] = np.where(lines["from_norm"] <= lines["to_norm"], lines["from_norm"], lines["to_norm"])
    lines["v"] = np.where(lines["from_norm"] <= lines["to_norm"], lines["to_norm"], lines["from_norm"])
    lines = lines.groupby(["Period", "u", "v"], as_index=False)["transmissionInstalledCap_MW"].max()

    periods = sorted(lines["Period"].unique().tolist(), key=parse_period_end_year)
    by_period: dict[str, pd.DataFrame] = {
        p: lines.loc[lines["Period"] == p, ["u", "v", "transmissionInstalledCap_MW"]].copy()
        for p in periods
    }
    # Rename for downstream compatibility
    for p in periods:
        by_period[p] = by_period[p].rename(columns={"transmissionInstalledCap_MW": "transmissionInstalledCap"})
    return by_period, periods


def map_generator_to_group(generator: str) -> str | None:
    g = str(generator).strip().lower()
    if "lignite" in g:            return "Lignite"
    if "hardcoal" in g or g == "hcoal" or "hcoal" in g: return "Hcoal"
    if "coal" in g:               return "Coal"
    if "gas" in g:                return "Gas"
    if "oil" in g:                return "Oil"
    if "bio" in g:                return "Bio"
    if "geo" in g:                return "Geo"
    if "nuclear" in g:            return "Nuclear"
    if "hydro" in g:              return "Hydro"
    if "wave" in g:               return "Wave"
    if "wind" in g:               return "Wind"
    if "solar" in g:              return "Solar"
    if "waste" in g:              return "Waste"
    return None


def parse_period_end_year(value: str) -> int:
    years = re.findall(r"\d{4}", str(value))
    return int(years[-1]) if years else -1


def load_capacity_by_period(results_path: Path) -> tuple[dict[str, pd.DataFrame], list[str]]:
    """Load installed generation capacity per period from results_output_gen.csv."""
    csv_file = results_path / "results_output_gen.csv"
    gen = pd.read_csv(csv_file)
    required = {"Node", "GeneratorType", "Period", "genInstalledCap_MW"}
    missing = required - set(gen.columns)
    if missing:
        raise ValueError(f"Missing columns in {csv_file}: {sorted(missing)}")

    gen["genInstalledCap_MW"] = pd.to_numeric(gen["genInstalledCap_MW"], errors="coerce").fillna(0.0)
    gen = gen.dropna(subset=["Node", "GeneratorType", "Period"]).copy()
    gen["Period"] = gen["Period"].astype(str)

    gen["TechGroup"] = gen["GeneratorType"].map(map_generator_to_group)
    unknown = sorted(gen.loc[gen["TechGroup"].isna(), "GeneratorType"].astype(str).unique())
    if unknown:
        print("Warning: Unmapped technologies skipped:", ", ".join(unknown))
    gen = gen.dropna(subset=["TechGroup"])

    cap_all = gen.groupby(["Period", "Node", "TechGroup"], as_index=False)["genInstalledCap_MW"].sum()
    print("Unique technologies in dataset:", ", ".join(sorted(gen["GeneratorType"].astype(str).unique())))

    periods = sorted(cap_all["Period"].unique().tolist(), key=parse_period_end_year)
    by_period: dict[str, pd.DataFrame] = {}
    for period in periods:
        cap = (
            cap_all.loc[cap_all["Period"] == period, ["Node", "TechGroup", "genInstalledCap_MW"]]
            .pivot(index="Node", columns="TechGroup", values="genInstalledCap_MW")
            .fillna(0.0).reset_index()
        )
        for tech in TECH_ORDER:
            if tech not in cap.columns:
                cap[tech] = 0.0
        cap["total_cap"] = cap[TECH_ORDER].sum(axis=1)
        cap["norm_node"] = cap["Node"].map(normalize_name)
        by_period[period] = cap
    return by_period, periods


def load_load_curtailment_last_period(results_path: Path) -> tuple[pd.DataFrame, str]:
    curtail_file = results_path / "results_output_Operational.csv"
    curtail = pd.read_csv(curtail_file)
    curtail["LoadShed_MW"] = pd.to_numeric(curtail["LoadShed_MW"], errors="coerce").fillna(0.0)
    periods = sorted(curtail["Period"].astype(str).unique().tolist(), key=parse_period_end_year)
    last_period = periods[-1]
    curtail_last = (
        curtail.loc[curtail["Period"].astype(str) == last_period, ["Node", "Scenario", "LoadShed_MW"]]
        .groupby(["Node", "Scenario"], as_index=False)["LoadShed_MW"].sum()
        .groupby("Node", as_index=False)["LoadShed_MW"].mean()
    )
    curtail_last["ExpectedLoadCurtailment_GWh"] = curtail_last["LoadShed_MW"] / 1000.0
    curtail_last = curtail_last[["Node", "ExpectedLoadCurtailment_GWh"]]
    curtail_last["norm_node"] = curtail_last["Node"].map(normalize_name)
    return curtail_last, last_period


def load_geojson_geometries(geojson_file: Path) -> list:
    with geojson_file.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    return [shape(feat["geometry"]) for feat in payload.get("features", []) if feat.get("geometry")]


def load_nuts2_country_borders(geojson_file: Path, country_codes: set[str]) -> list:
    with geojson_file.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    borders = []
    for feat in payload.get("features", []):
        geom = feat.get("geometry")
        props = feat.get("properties", {})
        if not geom:
            continue
        cntr = str(props.get("CNTR_CODE", "")).strip().upper()
        try:
            level = int(props.get("LEVL_CODE"))
        except (TypeError, ValueError):
            level = None
        if cntr in country_codes and level == 2:
            borders.append(shape(geom))
    return borders


def build_change_dataframe(first_cap: pd.DataFrame, last_cap: pd.DataFrame, group_techs: list[str]) -> pd.DataFrame:
    first = first_cap[["norm_node", *group_techs]].copy()
    last  = last_cap[["norm_node", *group_techs]].copy()
    first["first_value"] = first[group_techs].sum(axis=1)
    last["last_value"]   = last[group_techs].sum(axis=1)
    merged = (
        first[["norm_node", "first_value"]]
        .merge(last[["norm_node", "last_value"]], on="norm_node", how="outer")
        .fillna(0.0)
    )
    merged["delta_mw"] = merged["last_value"] - merged["first_value"]
    return merged[["norm_node", "delta_mw"]]

print("Helper functions loaded.")


In [ ]:
# ── Load all data ─────────────────────────────────────────
coords            = load_coords(COORDS_FILE)
offshore_norm_nodes = load_offshore_nodes(OFFSHORE_TAB_FILE)
offshore_to_onshore = build_offshore_to_onshore_map(coords, offshore_norm_nodes)
capacity_by_period, gen_periods   = load_capacity_by_period(RESULTS_PATH)
transmission_by_period, line_periods = load_transmission_by_period(RESULTS_PATH)
periods = sorted(set(gen_periods).union(line_periods))

print(f"Periods found: {periods}")
print(f"Offshore nodes: {len(offshore_norm_nodes)}")
print(f"Offshore → onshore map: {offshore_to_onshore}")


In [ ]:
# ── Plot: installed capacity pies (one figure per period) ─
coord_lookup = coords[["norm_node", "Latitude", "Longitude"]].drop_duplicates("norm_node")

for period in periods:
    if period not in capacity_by_period:
        print(f"Period {period}: no generation data, skipping.")
        continue

    capacity = capacity_by_period[period]
    if AGGREGATE_OFFSHORE_TO_CLOSEST_ONSHORE:
        capacity = aggregate_offshore_to_nearest_onshore(capacity, coords, offshore_norm_nodes, offshore_to_onshore)

    merged = capacity.merge(coord_lookup, on="norm_node", how="left")
    n_missing = merged["Latitude"].isna().sum()
    if n_missing:
        print(f"Period {period}: {n_missing} nodes without coordinates skipped:", merged.loc[merged["Latitude"].isna(), "Node"].tolist())
    merged = merged.dropna(subset=["Latitude", "Longitude"]).copy()
    if merged.empty:
        continue

    lines = transmission_by_period.get(period, pd.DataFrame(columns=["u", "v", "transmissionInstalledCap"])).copy()
    if not lines.empty and AGGREGATE_OFFSHORE_TO_CLOSEST_ONSHORE:
        lines["u"] = lines["u"].map(offshore_to_onshore).fillna(lines["u"])
        lines["v"] = lines["v"].map(offshore_to_onshore).fillna(lines["v"])
        lines = lines[lines["u"] != lines["v"]].copy()
        lines["a"] = np.where(lines["u"] <= lines["v"], lines["u"], lines["v"])
        lines["b"] = np.where(lines["u"] <= lines["v"], lines["v"], lines["u"])
        lines = lines.groupby(["a", "b"], as_index=False)["transmissionInstalledCap"].sum()
        lines = lines.rename(columns={"a": "u", "b": "v"})
    lines = (
        lines.merge(coord_lookup, left_on="u", right_on="norm_node", how="left")
        .rename(columns={"Latitude": "from_lat", "Longitude": "from_lon"}).drop(columns=["norm_node"])
        .merge(coord_lookup, left_on="v", right_on="norm_node", how="left")
        .rename(columns={"Latitude": "to_lat", "Longitude": "to_lon"}).drop(columns=["norm_node"])
    )
    lines = lines.dropna(subset=["from_lat", "from_lon", "to_lat", "to_lon"]).copy()

    # ── draw ──────────────────────────────────────────────
    projection = ccrs.PlateCarree()
    fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={"projection": projection})
    ax.add_feature(cfeature.OCEAN, facecolor="#dbeeff", zorder=0)
    ax.add_feature(cfeature.LAND, facecolor="#f8f8f8", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#666666", zorder=1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="#888888", zorder=1)
    if BASEMAP_FILE.exists():
        ax.add_geometries(load_geojson_geometries(BASEMAP_FILE), crs=GEOJSON_SOURCE_CRS,
                          facecolor="#ededed", edgecolor="#b6b6b6", linewidth=0.2, zorder=1)
        de_dk = load_nuts2_country_borders(BASEMAP_FILE, {"DE", "DK"})
        if de_dk:
            ax.add_geometries(de_dk, crs=GEOJSON_SOURCE_CRS, facecolor="none", edgecolor="#6B6B6B", linewidth=0.4, zorder=6)

    cap_vals = merged["total_cap"].clip(lower=0)
    max_cap  = cap_vals.max() if len(cap_vals) else 0.0
    radii = 0.15 + 1.2 * np.sqrt(cap_vals / max_cap) if max_cap > 0 else np.full(len(merged), 0.15)

    if not lines.empty:
        max_lc = float(lines["transmissionInstalledCap"].max())
        for _, line in lines.iterrows():
            lw = 0.2 + 2.8 * np.sqrt(float(line["transmissionInstalledCap"]) / max_lc) if max_lc > 0 else 0.2
            ax.plot([line["from_lon"], line["to_lon"]], [line["from_lat"], line["to_lat"]],
                    color="#2F5D62", linewidth=lw, alpha=0.8, transform=projection, zorder=4)

    for (_, row), radius in zip(merged.iterrows(), radii):
        total = float(row["total_cap"])
        if total <= 0:
            continue
        theta1 = 0.0
        for tech in TECH_ORDER:
            val = float(row[tech])
            if val <= 0:
                continue
            theta2 = theta1 + (val / total) * 360.0
            ax.add_patch(Wedge((row["Longitude"], row["Latitude"]), float(radius),
                               float(theta1), float(theta2),
                               facecolor=TECH_COLORS[tech], edgecolor="black", linewidth=0.25,
                               transform=projection, zorder=5))
            theta1 = theta2

    for _, row in merged.nlargest(12, "total_cap").iterrows():
        ax.text(row["Longitude"] + 0.15, row["Latitude"] + 0.15, row["Node"],
                fontsize=8, transform=projection, zorder=6)

    ax.set_extent([merged["Longitude"].min()-4, merged["Longitude"].max()+4,
                   merged["Latitude"].min()-3,  merged["Latitude"].max()+3], crs=projection)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4, linestyle="--", zorder=2)
    gl.top_labels = False; gl.right_labels = False
    ax.set_title(f"North Sea case: installed generation capacity by node\nInvestment period: {period}")

    tech_legend = [Patch(facecolor=TECH_COLORS[t], edgecolor="black", label=t) for t in TECH_ORDER]
    leg1 = ax.legend(handles=tech_legend, loc="lower left", fontsize=8, title="Technology group", title_fontsize=9, framealpha=0.95)
    ax.add_artist(leg1)
    if max_cap > 0:
        size_vals = [max_cap * f for f in [0.1, 0.4, 1.0]]
        size_handles = [Line2D([0],[0], marker="o", color="none", markerfacecolor="white",
                                markeredgecolor="black", markeredgewidth=0.8,
                                markersize=(0.15+1.2*np.sqrt(v/max_cap))*9.5, label=f"{v:,.0f} MW")
                        for v in size_vals]
        ax.legend(handles=[Line2D([0],[0], color="#2F5D62", lw=2, label="Transmission"), *size_handles],
                  loc="lower right", title="Capacity scale", title_fontsize=9, fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f'gen_capacity_{period}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Period {period}: {len(merged)} nodes plotted.")


In [ ]:
# ── Plot: installed-capacity change bars (first → last period) ─
period_first = gen_periods[0]
period_last  = gen_periods[-1]

# Print which actual generator types in the dataset fall into each change group
_gen_raw = pd.read_csv(RESULTS_PATH / 'results_output_gen.csv')
_gen_raw['TechGroup'] = _gen_raw['GeneratorType'].map(map_generator_to_group)
print('Technologies by change group:')
for _gname, _gtechs in CHANGE_GROUPS.items():
    _matched = sorted(_gen_raw.loc[_gen_raw['TechGroup'].isin(_gtechs), 'GeneratorType'].unique())
    print(f'  {_gname}: {_matched}')

if period_first in capacity_by_period and period_last in capacity_by_period:
    cap_first = capacity_by_period[period_first]
    cap_last  = capacity_by_period[period_last]
    if AGGREGATE_OFFSHORE_TO_CLOSEST_ONSHORE:
        cap_first = aggregate_offshore_to_nearest_onshore(cap_first, coords, offshore_norm_nodes, offshore_to_onshore)
        cap_last  = aggregate_offshore_to_nearest_onshore(cap_last,  coords, offshore_norm_nodes, offshore_to_onshore)

    for group_label, techs in CHANGE_GROUPS.items():
        delta_df = build_change_dataframe(cap_first, cap_last, techs)
        plot_df = (
            delta_df.merge(coords[["norm_node", "Location", "Latitude", "Longitude"]], on="norm_node", how="left")
            .dropna(subset=["Latitude", "Longitude"]).copy()
        )
        if plot_df.empty:
            print(f"No coordinates for group '{group_label}', skipping.")
            continue

        max_abs = float(plot_df["delta_mw"].abs().max()) or 1.0
        min_h, max_h = 0.08, 1.5

        projection = ccrs.PlateCarree()
        fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={"projection": projection})
        ax.add_feature(cfeature.OCEAN, facecolor="#dbeeff", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="#f8f8f8", zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#666666", zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="#888888", zorder=1)
        if BASEMAP_FILE.exists():
            ax.add_geometries(load_geojson_geometries(BASEMAP_FILE), crs=GEOJSON_SOURCE_CRS,
                              facecolor="#ededed", edgecolor="#b6b6b6", linewidth=0.2, zorder=1)
            de_dk = load_nuts2_country_borders(BASEMAP_FILE, {"DE", "DK"})
            if de_dk:
                ax.add_geometries(de_dk, crs=GEOJSON_SOURCE_CRS, facecolor="none",
                                  edgecolor="#6B6B6B", linewidth=0.4, zorder=3)

        for _, row in plot_df.iterrows():
            delta = float(row["delta_mw"])
            if abs(delta) < 1e-9:
                continue
            bar_h = min_h + (max_h - min_h) * np.sqrt(abs(delta) / max_abs)
            y0 = float(row["Latitude"])
            ax.plot([row["Longitude"], row["Longitude"]], [y0, y0 + (bar_h if delta > 0 else -bar_h)],
                    color="#1a9850" if delta > 0 else "#d73027",
                    linewidth=3.6, transform=projection, zorder=4)

        for _, row in plot_df.loc[plot_df["delta_mw"].abs().nlargest(10).index].iterrows():
            ax.text(row["Longitude"]+0.12, row["Latitude"]+0.12, str(row["Location"]),
                    fontsize=7.5, transform=projection, zorder=5)

        ax.set_extent([plot_df["Longitude"].min()-4, plot_df["Longitude"].max()+4,
                       plot_df["Latitude"].min()-3,  plot_df["Latitude"].max()+3], crs=projection)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4, linestyle="--", zorder=2)
        gl.top_labels = False; gl.right_labels = False
        ax.set_title(f"North Sea case: capacity change by node\n{group_label.replace('_',' ').title()} ({period_first} → {period_last})")

        size_vals = [0.25*max_abs, 0.6*max_abs, max_abs]
        legend_handles = [
            Line2D([0],[0], color="#1a9850", lw=3.6, label="Increase"),
            Line2D([0],[0], color="#d73027", lw=3.6, label="Decrease"),
            *[Line2D([0],[0], color="#444", lw=max(2.0, 3.6*(min_h+(max_h-min_h)*np.sqrt(v/max_abs))/max_h),
                     label=f"{v:,.0f} MW") for v in size_vals],
        ]
        ax.legend(handles=legend_handles, loc="lower right", title="Bar meaning", title_fontsize=9, fontsize=8)
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f'capacity_change_{group_label}.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
# ── Plot: expected load curtailment (last period) ─────────
curtailment, curtail_period = load_load_curtailment_last_period(RESULTS_PATH)

if AGGREGATE_OFFSHORE_TO_CLOSEST_ONSHORE and not curtailment.empty:
    curtailment["target_norm_node"] = curtailment["norm_node"].map(offshore_to_onshore).fillna(curtailment["norm_node"])
    curtailment = (
        curtailment.groupby("target_norm_node", as_index=False)["ExpectedLoadCurtailment_GWh"]
        .sum().rename(columns={"target_norm_node": "norm_node"})
    )

plot_df = (
    curtailment.merge(coords[["norm_node", "Location", "Latitude", "Longitude"]], on="norm_node", how="left")
    .dropna(subset=["Latitude", "Longitude"]).copy()
)

values = plot_df["ExpectedLoadCurtailment_GWh"].clip(lower=0.0)
vmax   = float(values.max()) if len(values) else 0.0
sizes  = 25 + 900 * np.sqrt(values / vmax) if vmax > 0 else np.full(len(plot_df), 25.0)

projection = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={"projection": projection})
ax.add_feature(cfeature.OCEAN, facecolor="#dbeeff", zorder=0)
ax.add_feature(cfeature.LAND, facecolor="#f8f8f8", zorder=0)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="#666666", zorder=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="#888888", zorder=1)
if BASEMAP_FILE.exists():
    ax.add_geometries(load_geojson_geometries(BASEMAP_FILE), crs=GEOJSON_SOURCE_CRS,
                      facecolor="#ededed", edgecolor="#b6b6b6", linewidth=0.2, zorder=1)
    de_dk = load_nuts2_country_borders(BASEMAP_FILE, {"DE", "DK"})
    if de_dk:
        ax.add_geometries(de_dk, crs=GEOJSON_SOURCE_CRS, facecolor="none",
                          edgecolor="#6B6B6B", linewidth=0.4, zorder=3)

sc = ax.scatter(plot_df["Longitude"], plot_df["Latitude"],
                s=sizes, c=values, cmap="magma", alpha=0.9,
                edgecolor="black", linewidth=0.35, transform=projection, zorder=5)
for _, row in plot_df.nlargest(12, "ExpectedLoadCurtailment_GWh").iterrows():
    ax.text(row["Longitude"]+0.12, row["Latitude"]+0.12, str(row["Location"]),
            fontsize=7.5, transform=projection, zorder=6)

ax.set_extent([plot_df["Longitude"].min()-4, plot_df["Longitude"].max()+4,
               plot_df["Latitude"].min()-3,  plot_df["Latitude"].max()+3], crs=projection)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.4, linestyle="--", zorder=2)
gl.top_labels = False; gl.right_labels = False
ax.set_title(f"North Sea case: expected annual curtailment by node\nPeriod: {curtail_period}")
cbar = plt.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label("Expected load curtailment [GWh]")
plt.tight_layout()
plt.savefig(PLOT_DIR / f'load_curtailment_{curtail_period}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Plot: transmission capacity change (first → last period) ─
# Load raw transmission CSV to compute per-link delta
_trans_raw = pd.read_csv(RESULTS_PATH / 'results_output_transmision.csv')
_trans_raw['transmissionInstalledCap_MW'] = pd.to_numeric(
    _trans_raw['transmissionInstalledCap_MW'], errors='coerce').fillna(0.0)
_trans_raw['Period'] = _trans_raw['Period'].astype(str)
_trans_raw['u'] = _trans_raw.apply(
    lambda r: normalize_name(r['BetweenNode']) if normalize_name(r['BetweenNode']) <= normalize_name(r['AndNode'])
              else normalize_name(r['AndNode']), axis=1)
_trans_raw['v'] = _trans_raw.apply(
    lambda r: normalize_name(r['AndNode']) if normalize_name(r['BetweenNode']) <= normalize_name(r['AndNode'])
              else normalize_name(r['BetweenNode']), axis=1)

# Deduplicate to max cap per link per period
_trans_raw = _trans_raw.groupby(['Period', 'u', 'v'], as_index=False)['transmissionInstalledCap_MW'].max()

_t_first = line_periods[0]
_t_last  = line_periods[-1]
print(f"Transmission change: {_t_first} → {_t_last}")

_first_links = _trans_raw.loc[_trans_raw['Period'] == _t_first, ['u', 'v', 'transmissionInstalledCap_MW']].rename(
    columns={'transmissionInstalledCap_MW': 'cap_first'})
_last_links  = _trans_raw.loc[_trans_raw['Period'] == _t_last,  ['u', 'v', 'transmissionInstalledCap_MW']].rename(
    columns={'transmissionInstalledCap_MW': 'cap_last'})

_delta_links = (
    _first_links.merge(_last_links, on=['u', 'v'], how='outer').fillna(0.0)
)
_delta_links['delta_MW'] = _delta_links['cap_last'] - _delta_links['cap_first']
_delta_links = _delta_links[_delta_links['delta_MW'].abs() > 1e-6].copy()

# Attach coordinates for both endpoints
_coord_lu = coords[['norm_node', 'Latitude', 'Longitude']].drop_duplicates('norm_node')
_delta_links = (
    _delta_links
    .merge(_coord_lu, left_on='u', right_on='norm_node', how='left')
    .rename(columns={'Latitude': 'lat_u', 'Longitude': 'lon_u'}).drop(columns=['norm_node'])
    .merge(_coord_lu, left_on='v', right_on='norm_node', how='left')
    .rename(columns={'Latitude': 'lat_v', 'Longitude': 'lon_v'}).drop(columns=['norm_node'])
    .dropna(subset=['lat_u', 'lon_u', 'lat_v', 'lon_v'])
)
print(f"Links with capacity change: {len(_delta_links)}")

if not _delta_links.empty:
    max_abs_trans = float(_delta_links['delta_MW'].abs().max())
    min_lw, max_lw = 0.5, 6.0

    projection = ccrs.PlateCarree()
    fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': projection})
    ax.add_feature(cfeature.OCEAN,     facecolor='#dbeeff', zorder=0)
    ax.add_feature(cfeature.LAND,      facecolor='#f8f8f8', zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#666666', zorder=1)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, edgecolor='#888888', zorder=1)
    if BASEMAP_FILE.exists():
        ax.add_geometries(load_geojson_geometries(BASEMAP_FILE), crs=GEOJSON_SOURCE_CRS,
                          facecolor='#ededed', edgecolor='#b6b6b6', linewidth=0.2, zorder=1)
        de_dk = load_nuts2_country_borders(BASEMAP_FILE, {'DE', 'DK'})
        if de_dk:
            ax.add_geometries(de_dk, crs=GEOJSON_SOURCE_CRS, facecolor='none',
                              edgecolor='#6B6B6B', linewidth=0.4, zorder=3)

    for _, lnk in _delta_links.iterrows():
        d = float(lnk['delta_MW'])
        lw = min_lw + (max_lw - min_lw) * np.sqrt(abs(d) / max_abs_trans)
        color = '#1a9850' if d > 0 else '#d73027'
        mid_lon = (lnk['lon_u'] + lnk['lon_v']) / 2
        mid_lat = (lnk['lat_u'] + lnk['lat_v']) / 2
        ax.plot([lnk['lon_u'], lnk['lon_v']], [lnk['lat_u'], lnk['lat_v']],
                color=color, linewidth=lw, alpha=0.85, transform=projection, zorder=4)
        ax.text(mid_lon, mid_lat, f"{d:+,.0f}", fontsize=5.5, ha='center', va='center',
                color='black', transform=projection, zorder=5)

    # Plot all nodes as small dots for reference
    all_nodes = pd.concat([
        _delta_links[['u', 'lat_u', 'lon_u']].rename(columns={'u': 'norm_node', 'lat_u': 'Latitude', 'lon_u': 'Longitude'}),
        _delta_links[['v', 'lat_v', 'lon_v']].rename(columns={'v': 'norm_node', 'lat_v': 'Latitude', 'lon_v': 'Longitude'}),
    ]).drop_duplicates('norm_node')
    ax.scatter(all_nodes['Longitude'], all_nodes['Latitude'], s=18, color='#333333',
               zorder=6, transform=projection)

    all_lons = pd.concat([_delta_links['lon_u'], _delta_links['lon_v']])
    all_lats = pd.concat([_delta_links['lat_u'], _delta_links['lat_v']])
    ax.set_extent([all_lons.min()-3, all_lons.max()+3, all_lats.min()-2, all_lats.max()+2], crs=projection)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--', zorder=2)
    gl.top_labels = False; gl.right_labels = False
    ax.set_title(f"North Sea case: transmission capacity change by link [MW]\n({_t_first} → {_t_last})")

    size_vals_t = [0.25*max_abs_trans, 0.6*max_abs_trans, max_abs_trans]
    legend_handles_t = [
        Line2D([0],[0], color='#1a9850', lw=3.5, label='Increase'),
        Line2D([0],[0], color='#d73027', lw=3.5, label='Decrease'),
        *[Line2D([0],[0], color='#555',
                 lw=min_lw+(max_lw-min_lw)*np.sqrt(v/max_abs_trans),
                 label=f'{v:,.0f} MW') for v in size_vals_t],
    ]
    ax.legend(handles=legend_handles_t, loc='lower right', title='Capacity change', title_fontsize=9, fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'transmission_capacity_change.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No transmission links with capacity change found between the two periods.")


In [ ]:
# ── Plot: transmission capacity change – percentage labels, width = final capacity ──
_trans2 = pd.read_csv(RESULTS_PATH / 'results_output_transmision.csv')
_trans2['transmissionInstalledCap_MW'] = pd.to_numeric(
    _trans2['transmissionInstalledCap_MW'], errors='coerce').fillna(0.0)
_trans2['Period'] = _trans2['Period'].astype(str)
_trans2['u'] = _trans2.apply(
    lambda r: normalize_name(r['BetweenNode']) if normalize_name(r['BetweenNode']) <= normalize_name(r['AndNode'])
              else normalize_name(r['AndNode']), axis=1)
_trans2['v'] = _trans2.apply(
    lambda r: normalize_name(r['AndNode']) if normalize_name(r['BetweenNode']) <= normalize_name(r['AndNode'])
              else normalize_name(r['BetweenNode']), axis=1)
_trans2 = _trans2.groupby(['Period', 'u', 'v'], as_index=False)['transmissionInstalledCap_MW'].max()

_t2_first = line_periods[0]
_t2_last  = line_periods[-1]
print(f"Transmission % change: {_t2_first} → {_t2_last}")

_f2 = _trans2.loc[_trans2['Period'] == _t2_first, ['u', 'v', 'transmissionInstalledCap_MW']].rename(
    columns={'transmissionInstalledCap_MW': 'cap_first'})
_l2 = _trans2.loc[_trans2['Period'] == _t2_last,  ['u', 'v', 'transmissionInstalledCap_MW']].rename(
    columns={'transmissionInstalledCap_MW': 'cap_last'})

_dl2 = _f2.merge(_l2, on=['u', 'v'], how='outer').fillna(0.0)
_dl2['delta_MW']  = _dl2['cap_last'] - _dl2['cap_first']
_dl2 = _dl2[_dl2['delta_MW'].abs() > 1e-6].copy()

# Percentage change label
def _pct_label(row):
    if row['cap_first'] < 1e-6:
        return 'New'
    if row['cap_last'] < 1e-6:
        return '-100%'
    pct = row['delta_MW'] / row['cap_first'] * 100.0
    return f"{pct:+.0f}%"

_dl2['pct_label'] = _dl2.apply(_pct_label, axis=1)

# Attach coordinates
_coord_lu2 = coords[['norm_node', 'Latitude', 'Longitude']].drop_duplicates('norm_node')
_dl2 = (
    _dl2
    .merge(_coord_lu2, left_on='u', right_on='norm_node', how='left')
    .rename(columns={'Latitude': 'lat_u', 'Longitude': 'lon_u'}).drop(columns=['norm_node'])
    .merge(_coord_lu2, left_on='v', right_on='norm_node', how='left')
    .rename(columns={'Latitude': 'lat_v', 'Longitude': 'lon_v'}).drop(columns=['norm_node'])
    .dropna(subset=['lat_u', 'lon_u', 'lat_v', 'lon_v'])
)
print(f"Links with capacity change: {len(_dl2)}")

if not _dl2.empty:
    max_cap2   = float(_dl2['cap_last'].max())
    min_lw2, max_lw2 = 0.5, 7.0

    projection2 = ccrs.PlateCarree()
    fig2, ax2 = plt.subplots(figsize=(14, 10), subplot_kw={'projection': projection2})
    ax2.add_feature(cfeature.OCEAN,     facecolor='#dbeeff', zorder=0)
    ax2.add_feature(cfeature.LAND,      facecolor='#f8f8f8', zorder=0)
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#666666', zorder=1)
    ax2.add_feature(cfeature.BORDERS,   linewidth=0.4, edgecolor='#888888', zorder=1)
    if BASEMAP_FILE.exists():
        ax2.add_geometries(load_geojson_geometries(BASEMAP_FILE), crs=GEOJSON_SOURCE_CRS,
                           facecolor='#ededed', edgecolor='#b6b6b6', linewidth=0.2, zorder=1)
        de_dk2 = load_nuts2_country_borders(BASEMAP_FILE, {'DE', 'DK'})
        if de_dk2:
            ax2.add_geometries(de_dk2, crs=GEOJSON_SOURCE_CRS, facecolor='none',
                               edgecolor='#6B6B6B', linewidth=0.4, zorder=3)

    for _, lnk2 in _dl2.iterrows():
        final_cap = float(lnk2['cap_last'])
        lw2 = min_lw2 + (max_lw2 - min_lw2) * np.sqrt(final_cap / max_cap2)
        color2 = '#1a9850' if float(lnk2['delta_MW']) > 0 else '#d73027'
        mid_lon2 = (lnk2['lon_u'] + lnk2['lon_v']) / 2
        mid_lat2 = (lnk2['lat_u'] + lnk2['lat_v']) / 2
        ax2.plot([lnk2['lon_u'], lnk2['lon_v']], [lnk2['lat_u'], lnk2['lat_v']],
                 color=color2, linewidth=lw2, alpha=0.85, transform=projection2, zorder=4)
        ax2.text(mid_lon2, mid_lat2, lnk2['pct_label'], fontsize=5.5, ha='center', va='center',
                 color='black', transform=projection2, zorder=5)

    # Node dots
    all_nodes2 = pd.concat([
        _dl2[['u', 'lat_u', 'lon_u']].rename(columns={'u': 'norm_node', 'lat_u': 'Latitude', 'lon_u': 'Longitude'}),
        _dl2[['v', 'lat_v', 'lon_v']].rename(columns={'v': 'norm_node', 'lat_v': 'Latitude', 'lon_v': 'Longitude'}),
    ]).drop_duplicates('norm_node')
    ax2.scatter(all_nodes2['Longitude'], all_nodes2['Latitude'], s=18, color='#333333',
                zorder=6, transform=projection2)

    all_lons2 = pd.concat([_dl2['lon_u'], _dl2['lon_v']])
    all_lats2 = pd.concat([_dl2['lat_u'], _dl2['lat_v']])
    ax2.set_extent([all_lons2.min()-3, all_lons2.max()+3, all_lats2.min()-2, all_lats2.max()+2], crs=projection2)
    gl2 = ax2.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--', zorder=2)
    gl2.top_labels = False; gl2.right_labels = False
    ax2.set_title(
        f"North Sea case: transmission capacity change [%] by link\n"
        f"(line width = final capacity; {_t2_first} → {_t2_last})"
    )

    size_refs = [0.25 * max_cap2, 0.6 * max_cap2, max_cap2]
    legend_handles2 = [
        Line2D([0],[0], color='#1a9850', lw=3.5, label='Increase'),
        Line2D([0],[0], color='#d73027', lw=3.5, label='Decrease'),
        *[Line2D([0],[0], color='#555',
                 lw=min_lw2 + (max_lw2 - min_lw2) * np.sqrt(v / max_cap2),
                 label=f'{v:,.0f} MW (final)') for v in size_refs],
    ]
    ax2.legend(handles=legend_handles2, loc='lower right', title='Line = final capacity\nColor = direction',
               title_fontsize=8.5, fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'transmission_capacity_final.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No transmission links with capacity change found.")


In [ ]:
# ── Plot: ALL transmission lines in the final period, coloured by % capacity change ──
# Line width  = final installed capacity
# Line colour = diverging spectrum: red (decrease) → grey (no change) → green (increase)

import matplotlib.colors as mcolors

_t3_first = line_periods[0]
_t3_last  = line_periods[-1]
print(f"Full transmission map: {_t3_first} → {_t3_last}")

# ── Build per-link table (first & last period) ─────────────────────────────
_trans3 = pd.read_csv(RESULTS_PATH / 'results_output_transmision.csv')
_trans3['transmissionInstalledCap_MW'] = pd.to_numeric(
    _trans3['transmissionInstalledCap_MW'], errors='coerce').fillna(0.0)
_trans3['Period'] = _trans3['Period'].astype(str)
_trans3['u'] = _trans3.apply(
    lambda r: normalize_name(r['BetweenNode']) if normalize_name(r['BetweenNode']) <= normalize_name(r['AndNode'])
              else normalize_name(r['AndNode']), axis=1)
_trans3['v'] = _trans3.apply(
    lambda r: normalize_name(r['AndNode']) if normalize_name(r['BetweenNode']) <= normalize_name(r['AndNode'])
              else normalize_name(r['BetweenNode']), axis=1)
_trans3 = _trans3.groupby(['Period', 'u', 'v'], as_index=False)['transmissionInstalledCap_MW'].max()

_f3 = _trans3.loc[_trans3['Period'] == _t3_first, ['u', 'v', 'transmissionInstalledCap_MW']].rename(
    columns={'transmissionInstalledCap_MW': 'cap_first'})
_l3 = _trans3.loc[_trans3['Period'] == _t3_last,  ['u', 'v', 'transmissionInstalledCap_MW']].rename(
    columns={'transmissionInstalledCap_MW': 'cap_last'})

_all3 = _f3.merge(_l3, on=['u', 'v'], how='outer').fillna(0.0)
_all3['delta_MW']  = _all3['cap_last'] - _all3['cap_first']

# % change: new links (cap_first ≈ 0) → +inf → clamp to +200 %; removed → -100 %
def _pct3(row):
    if row['cap_first'] < 1.0:
        return 200.0 if row['cap_last'] > 1.0 else 0.0
    return row['delta_MW'] / row['cap_first'] * 100.0

_all3['pct_change'] = _all3.apply(_pct3, axis=1)

# ── Attach coordinates ─────────────────────────────────────────────────────
_coord_lu3 = coords[['norm_node', 'Latitude', 'Longitude']].drop_duplicates('norm_node')
_all3 = (
    _all3
    .merge(_coord_lu3, left_on='u', right_on='norm_node', how='left')
    .rename(columns={'Latitude': 'lat_u', 'Longitude': 'lon_u'}).drop(columns=['norm_node'])
    .merge(_coord_lu3, left_on='v', right_on='norm_node', how='left')
    .rename(columns={'Latitude': 'lat_v', 'Longitude': 'lon_v'}).drop(columns=['norm_node'])
    .dropna(subset=['lat_u', 'lon_u', 'lat_v', 'lon_v'])
)

# Only keep links with positive final capacity (skip decommissioned + never-built)
_all3 = _all3[_all3['cap_last'] > 1.0].copy()
print(f"Total links plotted: {len(_all3)}")

# ── Colour mapping ──────────────────────────────────────────────────────────
# Clamp % change to [-100, +200] then map to [0, 1] for the colormap
_pct_min, _pct_max = -100.0, 200.0
_cmap = mcolors.LinearSegmentedColormap.from_list(
    'trans_pct',
    [
        (0.0,  '#d73027'),   # -100 %  → red
        (0.33, '#fee08b'),   # ~0 %    → yellow
        (0.40, '#d9ef8b'),   # small + → light green
        (1.0,  '#1a9850'),   # +200 %  → dark green
    ]
)

def _pct_to_colour(pct):
    t = (max(_pct_min, min(_pct_max, pct)) - _pct_min) / (_pct_max - _pct_min)
    return _cmap(t)

max_cap3   = float(_all3['cap_last'].max())
min_lw3, max_lw3 = 0.4, 7.0

# ── Draw ───────────────────────────────────────────────────────────────────
projection3 = ccrs.PlateCarree()
fig3, ax3 = plt.subplots(figsize=(14, 10), subplot_kw={'projection': projection3})
ax3.add_feature(cfeature.OCEAN,     facecolor='#dbeeff', zorder=0)
ax3.add_feature(cfeature.LAND,      facecolor='#f8f8f8', zorder=0)
ax3.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='#666666', zorder=1)
ax3.add_feature(cfeature.BORDERS,   linewidth=0.4, edgecolor='#888888', zorder=1)
if BASEMAP_FILE.exists():
    ax3.add_geometries(load_geojson_geometries(BASEMAP_FILE), crs=GEOJSON_SOURCE_CRS,
                       facecolor='#ededed', edgecolor='#b6b6b6', linewidth=0.2, zorder=1)
    de_dk3 = load_nuts2_country_borders(BASEMAP_FILE, {'DE', 'DK'})
    if de_dk3:
        ax3.add_geometries(de_dk3, crs=GEOJSON_SOURCE_CRS, facecolor='none',
                           edgecolor='#6B6B6B', linewidth=0.4, zorder=3)

for _, lnk3 in _all3.iterrows():
    lw3    = min_lw3 + (max_lw3 - min_lw3) * np.sqrt(float(lnk3['cap_last']) / max_cap3)
    color3 = _pct_to_colour(float(lnk3['pct_change']))
    ax3.plot([lnk3['lon_u'], lnk3['lon_v']], [lnk3['lat_u'], lnk3['lat_v']],
             color=color3, linewidth=lw3, alpha=0.85, transform=projection3, zorder=4)

# Node dots
_all_nodes3 = pd.concat([
    _all3[['u', 'lat_u', 'lon_u']].rename(columns={'u': 'norm_node', 'lat_u': 'Latitude', 'lon_u': 'Longitude'}),
    _all3[['v', 'lat_v', 'lon_v']].rename(columns={'v': 'norm_node', 'lat_v': 'Latitude', 'lon_v': 'Longitude'}),
]).drop_duplicates('norm_node')
ax3.scatter(_all_nodes3['Longitude'], _all_nodes3['Latitude'], s=14, color='#222222',
            zorder=6, transform=projection3)

all_lons3 = pd.concat([_all3['lon_u'], _all3['lon_v']])
all_lats3 = pd.concat([_all3['lat_u'], _all3['lat_v']])
ax3.set_extent([all_lons3.min()-3, all_lons3.max()+3, all_lats3.min()-2, all_lats3.max()+2], crs=projection3)
gl3 = ax3.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--', zorder=2)
gl3.top_labels = False; gl3.right_labels = False
ax3.set_title(
    f"Transmission – final installed capacity with % change\n"
    f"(line width = final capacity; colour = % change, {_t3_first} → {_t3_last})"
)

# ── Colorbar ───────────────────────────────────────────────────────────────
sm3 = plt.cm.ScalarMappable(cmap=_cmap, norm=mcolors.Normalize(vmin=_pct_min, vmax=_pct_max))
sm3.set_array([])
cbar3 = plt.colorbar(sm3, ax=ax3, shrink=0.55, pad=0.02, orientation='vertical')
cbar3.set_label('Capacity change [%]', fontsize=9)
cbar3.set_ticks([-100, -50, 0, 50, 100, 150, 200])
cbar3.set_ticklabels(['-100%', '-50%', '0%', '+50%', '+100%', '+150%', '≥+200% (new)'])

# ── Line-width legend ──────────────────────────────────────────────────────
_ref_caps = [0.25 * max_cap3, 0.5 * max_cap3, max_cap3]
_size_handles3 = [
    Line2D([0], [0], color='#555', lw=min_lw3 + (max_lw3 - min_lw3) * np.sqrt(v / max_cap3),
           label=f'{v:,.0f} MW')
    for v in _ref_caps
]
ax3.legend(handles=_size_handles3, loc='lower left', title='Final capacity', title_fontsize=9, fontsize=8)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'transmission_final_capacity_pct_change.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {PLOT_DIR / 'transmission_final_capacity_pct_change.png'}")